In [2]:
import json
import pandas as pd



In [5]:
def flatten_posts_dict(posts_by_tag):
    flat_posts = {}
    for tag, posts in posts_by_tag.items():
        for uri, post in posts.items():
            post = dict(post)          
            post["hashtag"] = tag      
            flat_posts[uri] = post
    return flat_posts

In [ ]:

with open("userdata_2.json", "r", encoding="utf-8") as f:
    raw_user_data = json.load(f)

with open("posts_2.json", "r", encoding="utf-8") as f:
    postss = json.load(f)

posts = flatten_posts_dict(postss)

In [7]:

def build_post_text_dict(posts, raw_user_data, csv_path):


    pairs_df = pd.read_csv(csv_path)

    post_text_dict = {}


    target_post_ids = set(pairs_df["P_id"].dropna().unique())

    for uri in target_post_ids:
        post = posts.get(uri)
        if post:
            text = post.get("text")
            if text:
                post_text_dict[uri] = text


    user_ids = set(pairs_df["A_id"].dropna().unique()) | \
               set(pairs_df["S_id"].dropna().unique())

    for user_id in user_ids:
        user_data = raw_user_data.get(user_id)
        if not user_data:
            continue

        for activity in user_data.get("history", []):
            uri = activity.get("post_uri")
            text = activity.get("text")

            if uri and text:
                post_text_dict[uri] = text

    return post_text_dict


In [8]:
all_text_dict = build_post_text_dict(posts,raw_user_data,"1_to_5.csv")

In [9]:
print(len(all_text_dict))

1987001


In [11]:
with open("text_of_posts.jsonl", "w") as f:
    for uri, text in all_text_dict.items():
        f.write(json.dumps({
            "post_uri": uri,
            "text": text
        }) + "\n")

print("JSONL created.")

JSONL created.
